# RAG Using LangChain by AI

## Complete Retrieval Augmented Generation System with YouTube Content

This notebook implements a full-featured RAG (Retrieval Augmented Generation) system that:

1. **Extracts** transcripts directly from YouTube videos
2. **Chunks** text into manageable segments for better processing
3. **Embeds** content using HuggingFace pre-trained models
4. **Stores** embeddings in FAISS vector database for fast retrieval
5. **Retrieves** relevant content based on semantic similarity
6. **Generates** intelligent answers using Groq's Llama model

Perfect for building intelligent Q&A systems over video content!

## Section 1: Install and Import Dependencies

In [14]:
!pip install -q youtube-transcript-api langchain-community langchain-groq \
               langchain-text-splitters sentence-transformers faiss-cpu python-dotenv


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import os
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

print("✅ All imports successful!")

✅ All imports successful!


## Section 2: Extract YouTube Transcript

Extract the full transcript from a YouTube video using its video ID.

In [ ]:
# YouTube Video ID - Replace with your video
# Example: "Gfr50f6ZBvo" (a 15-minute tech talk)
# You can extract the ID from: https://www.youtube.com/watch?v={VIDEO_ID}

video_id = "Gfr50f6ZBvo"

print(f"Fetching transcript for video ID: {video_id}")

try:
    # Create an instance of the API client
    api = YouTubeTranscriptApi()
    
    # List available transcripts
    transcript_list = api.list(video_id)
    
    # Get the first available transcript (try English first)
    transcript = transcript_list.find_transcript(['en'])
    
    # Fetch the transcript data
    transcript_data = transcript.fetch()
    
    # Extract text from each chunk/snippet
    transcript_text = " ".join([chunk.text for chunk in transcript_data])
    
    print(f"✅ Transcript extracted successfully!")
    print(f"📊 Total length: {len(transcript_text)} characters")
    print(f"📝 First 500 characters:\n{transcript_text[:500]}...")
    
except Exception as e:
    print(f"❌ Error fetching transcript: {e}")
    print("Make sure the video has captions/subtitles enabled")

Fetching transcript for video ID: Gfr50f6ZBvo
✅ Transcript extracted successfully!
📊 Total length: 373635 characters
📝 First 500 characters:
FetchedTranscriptSnippet(text='the following is a conversation with', start=0.08, duration=3.44) FetchedTranscriptSnippet(text='demus hasabis', start=1.76, duration=4.96) FetchedTranscriptSnippet(text='ceo and co-founder of deepmind', start=3.52, duration=5.119) FetchedTranscriptSnippet(text='a company that has published and builds', start=6.72, duration=4.48) FetchedTranscriptSnippet(text='some of the most incredible artificial', start=8.639, duration=4.561) FetchedTranscriptSnippet(text='intel...


## Section 3: Split Text into Chunks

Break the transcript into smaller chunks for embedding. This helps create more focused vectors.

In [ ]:
# Initialize the text splitter
# chunk_size: number of characters per chunk (1000 is good for semantic meaning)
# chunk_overlap: overlap between chunks (helps preserve context)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

# Create document chunks
documents = text_splitter.create_documents([transcript_text])

print(f"✅ Text splitting complete!")
print(f"📦 Number of chunks: {len(documents)}")
print(f"📏 Average chunk size: {len(transcript_text) // len(documents)} characters")
print(f"\n📄 Sample chunk (first chunk):\n{documents[0].page_content[:300]}...")

## Section 4: Generate Embeddings with HuggingFace

Convert text chunks into vector embeddings using a pre-trained model. This enables semantic similarity search.

In [ ]:
print("🚀 Loading HuggingFace embeddings model (this may take a moment on first run)...")

# Initialize embeddings using a pre-trained sentence transformer model
# all-MiniLM-L6-v2: Fast, lightweight model good for most tasks (~90MB)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

print("✅ Embeddings model loaded successfully!")
print(f"🔢 Embedding dimension: 384 (for all-MiniLM-L6-v2)")

# Test embedding a sample text
sample_text = "Machine learning and artificial intelligence"
sample_embedding = embeddings.embed_query(sample_text)
print(f"📊 Sample embedding shape: {len(sample_embedding)}")

## Section 5: Create and Store Vector Database with FAISS

Build a FAISS vector database from embedded chunks. FAISS enables fast similarity search.

In [ ]:
print("📚 Building FAISS vector store from embeddings...")

# Create FAISS vector store from documents
vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)

print("✅ FAISS vector store created successfully!")
print(f"📊 Total vectors stored: {vector_store.index.ntotal}")

# Test the vector store with a sample query
sample_query = "What is the main topic discussed?"
test_results = vector_store.similarity_search(sample_query, k=2)

print(f"\n🔍 Testing retrieval with sample query: '{sample_query}'")
print(f"✅ Found {len(test_results)} similar documents")
if test_results:
    print(f"\n📄 Most relevant chunk:\n{test_results[0].page_content[:200]}...")

## Section 6: Build Retriever Component

Create a retriever that will fetch the most relevant chunks for any query.

In [ ]:
# Create a retriever from the vector store
# search_type: "similarity" uses cosine similarity
# search_kwargs: {"k": 4} means retrieve top 4 most relevant chunks

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("✅ Retriever created successfully!")
print(f"📋 Retriever configuration: Returns top 4 most similar documents")

# Test the retriever
test_query = "What are the key points discussed?"
retrieved_docs = retriever.invoke(test_query)

print(f"\n🔍 Testing retriever with query: '{test_query}'")
print(f"✅ Retrieved {len(retrieved_docs)} documents")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n📄 Document {i}:\n{doc.page_content[:150]}...")

## Section 7: Set Up LLM with Groq

Initialize the Groq LLM for generating answers. Groq provides fast inference for large language models.

In [ ]:
# Set Groq API key - Get yours from https://console.groq.com
# IMPORTANT: Never commit API keys to version control!
# Uncomment and set your API key, or use environment variables

# Option 1: Set via environment variable
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"  # Replace with your actual key

# Option 2: Load from .env file (recommended for local development)
# load_dotenv()
# os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Initialize Groq LLM
# llama-3.3-70b-versatile: Powerful, accurate model for complex reasoning
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,  # Balance between creativity (1.0) and consistency (0)
    max_tokens=2048
)

print("✅ Groq LLM initialized successfully!")
print("🤖 Model: llama-3.3-70b-versatile")
print("🌡️ Temperature: 0.7 (creative but consistent)")

## Section 8: Create Prompt Template

Define how context and questions are formatted before sending to the LLM.

In [ ]:
# Create a prompt template that structures the input to the LLM
# This template ensures the LLM stays focused on the provided context

prompt_template = PromptTemplate(
    template="""You are a helpful assistant answering questions based on a video transcript.

CONTEXT (from the video):
{context}

QUESTION:
{question}

ANSWER:
Based ONLY on the provided context from the video transcript, answer the question above.
If the context doesn't contain relevant information to answer the question, say "I don't have enough information from the video to answer this question."
Be concise and direct in your answer.""",
    
    input_variables=["context", "question"]
)

print("✅ Prompt template created successfully!")
print("\n📋 Template structure:")
print("   - Uses context from retrieved documents")
print("   - Keeps LLM focused on video content")
print("   - Handles out-of-context questions gracefully")

## Section 9: Implement RAG Chain

Build the complete RAG pipeline connecting retriever, prompt, and LLM.

In [ ]:
# Helper function to format retrieved documents into readable context
def format_docs(docs):
    """Convert list of documents into formatted context string."""
    return "\n\n".join([doc.page_content for doc in docs])

# Build the RAG chain using LangChain's pipe operator
# Chain flow:
# 1. RunnableParallel: Fetch context from retriever & pass through question
# 2. prompt_template: Format context and question into the prompt
# 3. llm: Generate answer using Groq
# 4. StrOutputParser: Extract the text response

rag_chain = (
    RunnableParallel({
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    })
    | prompt_template
    | llm
    | StrOutputParser()
)

print("✅ RAG chain built successfully!")
print("\n🔗 Chain pipeline:")
print("   1. Query → Retriever (find relevant chunks)")
print("   2. Context + Question → Prompt Template")
print("   3. Formatted Prompt → Groq LLM")
print("   4. LLM Output → Parse as String")
print("\n✨ Ready to answer questions about the video!")

## Section 10: Test Query and Generation

Test the complete RAG system with example questions.

In [ ]:
# Test Query 1: General summary
print("=" * 80)
print("TEST QUERY 1: Video Summary")
print("=" * 80)

query1 = "Can you provide a summary of the main topics discussed in this video?"
print(f"❓ Question: {query1}\n")
answer1 = rag_chain.invoke(query1)
print(f"✅ Answer:\n{answer1}\n")

In [ ]:
# Test Query 2: Specific details
print("=" * 80)
print("TEST QUERY 2: Key Details")
print("=" * 80)

query2 = "What are the most important points or key takeaways mentioned?"
print(f"❓ Question: {query2}\n")
answer2 = rag_chain.invoke(query2)
print(f"✅ Answer:\n{answer2}\n")

In [ ]:
# Test Query 3: Follow-up question
print("=" * 80)
print("TEST QUERY 3: Detailed Explanation")
print("=" * 80)

query3 = "Explain the main concepts in more detail"
print(f"❓ Question: {query3}\n")
answer3 = rag_chain.invoke(query3)
print(f"✅ Answer:\n{answer3}\n")

## Interactive Query Interface

Use this section to ask your own questions about the video. Modify the `user_query` variable and run the cell.

In [ ]:
# 🎯 INTERACTIVE QUERY - Modify this to ask your own questions!
user_query = "What is the core message of this video?"

print("=" * 80)
print("YOUR CUSTOM QUERY")
print("=" * 80)
print(f"❓ Question: {user_query}\n")

try:
    answer = rag_chain.invoke(user_query)
    print(f"✅ Answer:\n{answer}\n")
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nMake sure your Groq API key is set correctly.")

## Advanced: Debug and Analysis

View the intermediate steps of the RAG pipeline for debugging and understanding.

In [ ]:
# Debug: See what documents are retrieved for a query
debug_query = "What are the key concepts?"

print("=" * 80)
print("DEBUG: Retrieved Documents")
print("=" * 80)
print(f"Query: {debug_query}\n")

retrieved_docs = retriever.invoke(debug_query)
print(f"Retrieved {len(retrieved_docs)} most relevant documents:\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"📄 Document {i}:")
    print(f"{'─' * 75}")
    print(doc.page_content[:400])
    print(f"{'─' * 75}\n")

## System Statistics and Information

In [ ]:
print("=" * 80)
print("📊 RAG SYSTEM STATISTICS")
print("=" * 80)

print(f"\n📹 Video Information:")
print(f"   • Video ID: {video_id}")
print(f"   • Transcript Length: {len(transcript_text):,} characters")
print(f"   • Estimated Words: {len(transcript_text) // 5:,}")

print(f"\n📚 Document Chunking:")
print(f"   • Total Chunks: {len(documents)}")
print(f"   • Chunk Size: 1000 characters")
print(f"   • Chunk Overlap: 200 characters")
print(f"   • Average Chunk Size: {sum(len(doc.page_content) for doc in documents) // len(documents)} characters")

print(f"\n🔢 Vector Database:")
print(f"   • Total Vectors: {vector_store.index.ntotal}")
print(f"   • Embedding Dimension: 384 (all-MiniLM-L6-v2)")
print(f"   • Retrieval Method: Cosine Similarity")
print(f"   • Top-k Results: 4")

print(f"\n🤖 LLM Configuration:")
print(f"   • Provider: Groq")
print(f"   • Model: llama-3.3-70b-versatile")
print(f"   • Temperature: 0.7")
print(f"   • Max Tokens: 2048")

print(f"\n✅ System Ready for Q&A!")
print("=" * 80)

## Usage Guide and Tips

### How to Use This Notebook:

1. **Set Your API Key**: Update the Groq API key in Section 7 (get one from https://console.groq.com)

2. **Change Video**: Replace the `video_id` in Section 2 with any YouTube video ID that has subtitles

3. **Ask Questions**: 
   - Use Section 10 (Interactive Query) to ask your own questions
   - Questions should relate to the video content
   - Be specific for better results

4. **Troubleshooting**:
   - If transcript fails: Make sure the video has captions enabled
   - If API fails: Check your Groq API key is correct
   - For slow retrieval: Reduce number of documents or chunk size

### Advanced Customizations:

- **Change Embedding Model**: Replace `sentence-transformers/all-MiniLM-L6-v2` with other models from HuggingFace
  - `all-mpnet-base-v2`: Better accuracy but slower (~420MB)
  - `all-distilroberta-v1`: Faster, smaller model

- **Tune Retrieval**: Modify `search_kwargs={"k": 4}` to return more/fewer documents

- **Adjust LLM Parameters**:
  - `temperature`: 0.0 (deterministic) to 1.0 (creative)
  - `max_tokens`: Increase for longer answers

### Supported Video Platforms:
- ✅ YouTube (with subtitles)
- ✅ Any video with transcript URL

---

**Created with ❤️ for AI-Powered Content Analysis**